In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, regularizers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import tensorflow_datasets as tfds

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
tf.get_logger().setLevel(logging.ERROR)

import dpmhm
from dpmhm.datasets import preprocessing, feature, utils, transformer

In [ ]:
TFDS_DATA_DIR = os.path.expanduser('~/tensorflow_datasets/')

_ = dpmhm.datasets.install('FEMTO', 
                            data_dir=TFDS_DATA_DIR,                           
                            manual_dir=os.path.expanduser('~/tmp/ieee-phm-2012-data-challenge-dataset-master/')
                          )

In [ ]:
batch_size = 64
n_embedding  = 256 
kernel_size = (3,3) 
tau = 0.1
projection_dim = 128 

In [ ]:
ds0 = tfds.load('FEMTO', split='train')

Plot the evolution of the vibrations from the bearing 1_1

In [ ]:
import re

eles = list(ds0.as_numpy_iterator())
sr = eles[1]['sampling_rate']['vibration']
L_numbers=[]
for el in eles:
    file=el['metadata']['FileName'].decode()
    parts = file.split('/')
    type_and_number = re.findall(r'(\w+)_(\d+)\.csv', parts[1])
    if type_and_number:
        type_part, number_part = type_and_number[0]
    if parts[0]=='Bearing1_1' and type_part=='acc':
        L_numbers.append(int(number_part))

X=np.zeros((max(L_numbers),len(eles[0]['signal']['vibration'][0])))
for el in eles:
    file=el['metadata']['FileName'].decode()
    parts = file.split('/')
    type_and_number = re.findall(r'(\w+)_(\d+)\.csv', parts[1])
    if type_and_number:
        type_part, number_part = type_and_number[0]
    if parts[0]=='Bearing1_1' and type_part=='acc':
        X[int(number_part)-1]=el['signal']['vibration'][0]

flattened = [item for sublist in X for item in sublist]

plt.figure(figsize=(20,5))
plt.plot(100*np.arange(len(flattened))/sr, flattened)
plt.xlabel('Time (s)')
plt.axhline(y=20, color='r', linestyle='--')
plt.axhline(y=-20, color='g', linestyle='--')
plt.title('Champ vibratoire du Bearing1_1')

In [ ]:
Bearings_names=['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']

Load the registered datasets (if does not exist, run creation_metadataset.ipynb)

In [ ]:
import json

outdir = Path('/volatile/home/bm279471/tmp/SA_femto')
os.makedirs(outdir, exist_ok=True)

ds_train = tf.data.Dataset.load(str(outdir/'ds_train'))
ds_val = tf.data.Dataset.load(str(outdir/'ds_val'))
ds_test = {}
labels_test={}
equivalence_labels_test={}

with open(outdir/'labels_train.json', 'r') as fp:
    labels_train = list(json.load(fp))
with open(outdir/'equivalence_labels_train.json', 'r') as fp:
    equivalence_labels_train= json.load(fp)

for file in Bearings_names:
    ds_test[file]=tf.data.Dataset.load(str(outdir/'ds_test_')+file)
    labels_dir='labels_test_'+file+'.json'
    with open(outdir/labels_dir, 'r') as fp:
        labels_test[file] = list(json.load(fp))
    equivalence_labels_dir='equivalence_labels_test_'+file+'.json'
    with open(outdir/equivalence_labels_dir, 'r') as fp:
        equivalence_labels_test[file] = json.load(fp)

In [ ]:
kernel_size=(3,3)
n_embedding=128
batch_size = 32
eles = list(ds_train.take(1).as_numpy_iterator())
input_shape = eles[0][0].shape
ds_size = utils.get_dataset_size(ds_train) + utils.get_dataset_size(ds_val) + utils.get_dataset_size(ds_test)

# Training of a ResNet

In [ ]:
ds_train_predict = ds_train.shuffle(5000, reshuffle_each_iteration=True).batch(32, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_predict = ds_val.shuffle(5000, reshuffle_each_iteration=True).batch(32, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

In [ ]:
Censor_factor_bis=0

def extract_E_bis(t):
    if tf.random.uniform([]) < Censor_factor_bis:
        rand=tf.random.uniform([], 0, t[0])
        round_rand=tf.round(rand.numpy()*10)
        return [round_rand.numpy()/ 10.0, t[1], 0.0]
    else:
        return [t[0], t[1], 1.0]

In [ ]:
el=list(ds_train_predict.take(1).as_numpy_iterator())
spec_shape_train=el[0][0].shape
lbl_shape_train=(32, np.shape(extract_E_bis([float(equivalence_labels_train[labels_train[el[0][1][0]]][0]), float(equivalence_labels_train[labels_train[el[0][1][0]]][1])]))[0])

def transform_labels_train_bis(labels):
    Labels=[]
    for lbl in labels:
        Labels.append(extract_E_bis([float(equivalence_labels_train[labels_train[lbl.numpy()-1]][0]), float(equivalence_labels_train[labels_train[lbl.numpy()-1]][1])]))
    return Labels

def generator_train_bis():
    for x, labels in ds_train_predict:
        yield x, transform_labels_train_bis(labels)

ds_train_p = tf.data.Dataset.from_generator(generator_train_bis,
                                                output_signature=(tf.TensorSpec(shape=spec_shape_train, dtype=tf.float32),
                                                                  tf.TensorSpec(shape=lbl_shape_train, dtype=tf.float32)))

el=list(ds_train_p.take(5).as_numpy_iterator())
for elem in el:
    print("shape spectrogram: ", elem[0].shape, "shape labels: ", elem[1].shape,"RUL:", elem[1][0][0], "Time elapsed:", elem[1][0][1], "Event indicator:", elem[1][0][2])

In [ ]:
el=list(ds_val_predict.take(1).as_numpy_iterator())
spec_shape_val=el[0][0].shape
lbl_shape_val=(32, np.shape(extract_E_bis([float(equivalence_labels_train[labels_train[el[0][1][0]]][0]), float(equivalence_labels_train[labels_train[el[0][1][0]]][1])]))[0])

def transform_labels_val(labels):
    Labels=[]
    for lbl in labels:
        Labels.append(extract_E_bis([float(equivalence_labels_train[labels_train[lbl.numpy()-1]][0]), float(equivalence_labels_train[labels_train[lbl.numpy()-1]][1])]))
    return Labels

def generator_val():
    for x, labels in ds_val_predict:
        yield x, transform_labels_val(labels)

ds_val_p = tf.data.Dataset.from_generator(generator_val,
                                                output_signature=(tf.TensorSpec(shape=spec_shape_val, dtype=tf.float32),
                                                                  tf.TensorSpec(shape=lbl_shape_val, dtype=tf.float32)))

el=list(ds_val_p.take(5).as_numpy_iterator())
for elem in el:
    print("shape spectrogram: ", elem[0].shape, "shape labels: ", elem[1].shape,"RUL:", elem[1][0][0], "Time elapsed:", elem[1][0][1], "Event indicator:", elem[1][0][2])

Train a ResNet to predict RUL values

In [ ]:
from keras.applications import ResNet50
from keras.layers import Dense, GlobalAveragePooling2D, ReLU, Input
from keras.models import Model

x = layers.Input((496, 8, 2))

adapt_model = keras.Sequential([
    layers.Flatten(name="flatten"),
    layers.Dense(4096, activation="relu", name="fc1"),
    layers.Dense(4096, activation="relu", name="fc2"),
    layers.Dense(1, activation=None, name="predictions")
])

y = adapt_model(ResNet50(weights=None, include_top=False, input_tensor=Input(shape=(496, 8, 2)))(x))

model = models.Model(x, y)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.MeanSquaredError(),
    metrics=['accuracy'],
)


early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=5,       
    restore_best_weights=True, 
    mode='min'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,         
    patience=2, 
    mode='min'    
)

model.fit(
    ds_train_p.repeat(), 
    epochs=30,
    validation_data=ds_val_p, 
    steps_per_epoch=utils.get_dataset_size(ds_train_p),
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
model.save_weights('ResNet.weights.h5')
model.load_weights('ResNet.weights.h5')

# Train the CoxPH model with features extracted from the ResNet

In [ ]:
ds_train_cox = ds_train.concatenate(ds_val).shuffle(5000, reshuffle_each_iteration=True).batch(1).prefetch(tf.data.AUTOTUNE)
ds_test_cox={}
for file in Bearings_names:
    ds_test_cox[file] = ds_test[file].batch(1)

In [ ]:
Censor_factor=0.25

def extract_E(t):
    if tf.random.uniform([]) < Censor_factor:
        rand=tf.random.uniform([], 0, t[0])
        round_rand=tf.round(rand.numpy()*10)
        return [round_rand.numpy()/ 10.0, t[1], 0.0]
    else:
        return [t[0], t[1], 1.0]

In [ ]:
el=list(ds_train_cox.take(1).as_numpy_iterator())
spec_shape_train=el[0][0].shape
lbl_shape_train=np.shape(extract_E([float(equivalence_labels_train[labels_train[el[0][1][0]]][0]), float(equivalence_labels_train[labels_train[el[0][1][0]]][1])]))

def transform_labels_train(lbl):
    return extract_E([float(equivalence_labels_train[labels_train[lbl[0].numpy()-1]][0]), float(equivalence_labels_train[labels_train[lbl[0].numpy()-1]][1])])

def generator_train():
    for x, lbl in ds_train_cox:
        yield x, transform_labels_train(lbl)

ds_train_CoxPH = tf.data.Dataset.from_generator(generator_train,
                                                output_signature=(tf.TensorSpec(shape=spec_shape_train, dtype=tf.float32),
                                                                  tf.TensorSpec(shape=lbl_shape_train, dtype=tf.float32)))

el=list(ds_train_CoxPH.take(5).as_numpy_iterator())
for elem in el:
    print("shape spectrogram: ", elem[0].shape, "RUL:", elem[1][0], "Time elapsed:", elem[1][1], "Event indicator:", elem[1][2])

In [ ]:
Censor_factor_test=1

def extract_E_test(t):
    if tf.random.uniform([]) < Censor_factor_test:
        rand=tf.random.uniform([], 0, t[0])
        round_rand=tf.round(rand.numpy()*10)
        return [round_rand.numpy()/ 10.0, t[1], 0.0]
    else:
        return [t[0], t[1], 1.0]

In [ ]:
el=list(ds_test_cox['Bearing1_3'].take(1).as_numpy_iterator())
spec_shape_test=el[0][0].shape
lbl_shape_test=np.shape(extract_E_test([float(equivalence_labels_test['Bearing1_3'][labels_test['Bearing1_3'][el[0][1][0]]][0]), float(equivalence_labels_test['Bearing1_3'][labels_test['Bearing1_3'][el[0][1][0]]][1])]))

ds_test_CoxPH={}
for file in ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']:
    def transform_labels_test(lbl):
        return extract_E_test([float(equivalence_labels_test[file][labels_test[file][lbl[0].numpy()-1]][0]), float(equivalence_labels_test[file][labels_test[file][lbl[0].numpy()-1]][1])])
    
    def generator_test():
        for x, lbl in ds_test_cox[file]:
            yield x, transform_labels_test(lbl)

    ds_test_CoxPH[file] = tf.data.Dataset.from_generator(generator_test,
                                                    output_signature=(tf.TensorSpec(shape=spec_shape_test, dtype=tf.float32),
                                                                    tf.TensorSpec(shape=lbl_shape_test, dtype=tf.float32)))

el=list(ds_test_CoxPH['Bearing1_3'].take(5).as_numpy_iterator())
for elem in el:
    print("shape spectrogram: ", elem[0].shape, "RUL:", elem[1][0], "Time elapsed:", elem[1][1], "Event indicator:", elem[1][2])

In [ ]:
ds_train_size=utils.get_dataset_size(ds_train_CoxPH)

RUL_train=np.zeros((ds_train_size))
Age_train=np.zeros((ds_train_size))
Event_train=np.zeros((ds_train_size))
for i, x in enumerate(list(ds_train_CoxPH.as_numpy_iterator())):
    RUL_train[i]=x[1][0]
    Age_train[i]=x[1][1]
    Event_train[i]=x[1][2]

RUL_test={}
Age_test={}
Event_test={}

for file in Bearings_names:
    ds_test_size=utils.get_dataset_size(ds_test_CoxPH[file])

    RUL_=np.zeros((ds_test_size))
    Age_=np.zeros((ds_test_size))
    Event_=np.zeros((ds_test_size))
    for i,x in enumerate(list(ds_test_CoxPH[file].as_numpy_iterator())):
        RUL_[i]=x[1][0]
        Age_[i]=x[1][1]
        Event_[i]=x[1][2]
    RUL_test[file]=RUL_
    Age_test[file]=Age_
    Event_test[file]=Event_    

RUL predicted by the ResNet

In [ ]:
RUL_resnet={}
for file in Bearings_names:
    RUL_resnet[file] = model.predict(ds_test_CoxPH[file])

ResNet without the last layer

In [ ]:
Cox_encoder=keras.Sequential()
for layer in model.layers[:-1]:
    Cox_encoder.add(layer)

for layer in model.layers[-1].layers[:-1]:
    Cox_encoder.add(layer)

Predict features for both train and test data

In [ ]:
features_train = Cox_encoder.predict(ds_train_CoxPH)

features_test={}
for file in Bearings_names:
    features_test[file] = Cox_encoder.predict(ds_test_CoxPH[file])

In [ ]:
import pandas as pd
from lifelines import CoxPHFitter
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Time of failure for each test bearing
Real_RUL=[23750, 16650, 28730, 28560, 26350, 19550, 8760, 26970, 8170, 2680, 5060]

non_null_columns = [a and b for a, b in zip(np.any(features_train != 0, axis=0),np.any(features_test[Bearings_names[0]] != 0, axis=0))]
transformed_features_train = features_train[:, non_null_columns]
data_train = pd.DataFrame(transformed_features_train, columns=[f'feature_{i}' for i in range(transformed_features_train.shape[1])])

# Compute the Variance Inflation Factor to remove the high colinearate columns
# vif_data = pd.DataFrame()
# vif_data["feature"] = data_train.columns
# vif_data["VIF"] = [variance_inflation_factor(data_train.values, i) for i in range(data_train.shape[1])]
# # Delete the high VIF values (superior to a tresh)
# high_vif_columns = vif_data[vif_data["VIF"] > sum(vif_data["VIF"])/len(vif_data["VIF"])]["feature"]
# data_train = data_train.drop(columns=high_vif_columns)

data_train['duration'] = RUL_train
data_train['event'] = Event_train.astype(int)
cph = CoxPHFitter(penalizer=0.0001)
cph.fit(data_train, duration_col='duration', event_col='event')

In [ ]:
Moy=[0, 0, 0, 0]
for j, file in enumerate(Bearings_names):    
    transformed_features_test = features_test[file][:, non_null_columns]

    data_test_1 = pd.DataFrame(transformed_features_test, columns=[f'feature_{i}' for i in range(transformed_features_test.shape[1])])
    # data_test_1 = data_test_1.drop(columns=high_vif_columns)
    data_test_1['duration'] = cph.predict_expectation(data_test_1).values
    data_test_1['event'] = np.ones_like(Event_test).astype(int)
    data_test_1['age'] = Age_test[file]

    plt.figure(figsize=(6, 4))
    plt.plot(cph.baseline_survival_)
    plt.title('CPH Baseline Survival Function')
    plt.xlabel('Time (s)')
    plt.ylabel('Survival Probability')
    plt.grid(True)
    plt.show()

    # predict RULs and plot them 
    combined = list(zip(Age_test[file], cph.predict_expectation(data_test_1.drop(columns=['duration', 'event', 'age'])), RUL_resnet[file]))
    sorted_combined = sorted(combined, key=lambda x: x[0])

    sorted_age_test, sorted_RUL_cph, sorted_RUL_resnet = zip(*sorted_combined)

    sorted_age_test = np.array(sorted_age_test)
    sorted_RUL_cph = np.array(sorted_RUL_cph)
    sorted_RUL_resnet = np.array(sorted_RUL_resnet)

    for i_, i in enumerate(sorted_age_test):
        sorted_RUL_cph[i_]+=abs(sorted_age_test[0]-sorted_age_test[-1])/2 - i

    plt.figure()
    plt.title('RUL values for the bearing '+file)
    plt.plot(sorted_age_test, sorted_RUL_resnet, label='Estimated RUL ResNet')
    plt.plot(sorted_age_test, sorted_RUL_cph, label='Estimated RUL Cox')
    plt.plot(sorted_age_test, [Real_RUL[j]- i for i in sorted_age_test], label='Real RUL')
    plt.xlabel('Age of the bearing at the time of register (s)')
    plt.ylabel('Estimated RUL of the sample (s)')
    plt.legend()

    #Compute C-index and MAPE for ResNet and CoxPH
    Concordantes_cph=0
    Discordantes_cph=0
    ExAequo_cph=0
    for i in range(len(sorted_age_test)-1):
        for k in range(i+1, len(sorted_age_test)):
            if sorted_RUL_cph[i]>sorted_RUL_cph[k]:
                Concordantes_cph+=1
            if sorted_RUL_cph[i]<sorted_RUL_cph[k]:
                Discordantes_cph+=1
            if sorted_RUL_cph[i]==sorted_RUL_cph[k]:
                ExAequo_cph+=1
    C_index_cph=(Concordantes_cph+0.5*ExAequo_cph)/(Concordantes_cph+Discordantes_cph+ExAequo_cph)

    Concordantes_res=0
    Discordantes_res=0
    ExAequo_res=0
    for i in range(len(sorted_age_test)-1):
        for k in range(i+1, len(sorted_age_test)):
            if sorted_RUL_resnet[i]>sorted_RUL_resnet[k]:
                Concordantes_res+=1
            if sorted_RUL_resnet[i]<sorted_RUL_resnet[k]:
                Discordantes_res+=1
            if sorted_RUL_resnet[i]==sorted_RUL_resnet[k]:
                ExAequo_res+=1
    C_index_res=(Concordantes_res+0.5*ExAequo_res)/(Concordantes_res+Discordantes_res+ExAequo_res)

    Moy[0]+=C_index_cph
    Moy[1]+=C_index_res
    Moy[2]+=np.mean(np.abs(([Real_RUL[j]- i for i in sorted_age_test] - sorted_RUL_resnet) / [Real_RUL[j]- i for i in sorted_age_test])) * 100
    Moy[3]+=np.mean(np.abs(([Real_RUL[j]- i for i in sorted_age_test] - sorted_RUL_cph) / [Real_RUL[j]- i for i in sorted_age_test])) * 100
    print("C-Index ResNet :", C_index_res*100,"%, C-Index CoxPH :", C_index_cph*100,"%")
    print("MAPE ResNet :", np.mean(np.abs(([Real_RUL[j]- i for i in sorted_age_test] - sorted_RUL_resnet) / [Real_RUL[j]- i for i in sorted_age_test])) * 100,"%, MAPE CoxPH :", np.mean(np.abs(([Real_RUL[j]- i for i in sorted_age_test] - sorted_RUL_cph) / [Real_RUL[j]- i for i in sorted_age_test])) * 100,"%")
print([m/11 for m in Moy])